<a href="https://colab.research.google.com/github/JSJeong-me/MCP/blob/main/02%20Agent%20Tool/4_mcp_module4_resources_prompts_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4 실습: MCP Resources & Prompts with FastMCP

이 노트북은 **Module 4: 서버 개발 실무 II - 리소스(Resources) 및 프롬프트(Prompts)** 실습용입니다.

학습 목표:
- Resources가 **읽기 전용 데이터 제공 방식**임을 이해한다.
- **동적 URI 템플릿**으로 맞춤형 데이터 접근을 구현한다.
- Prompts가 **재사용 가능한 지침 템플릿**임을 이해한다.
- FastMCP에서 `@mcp.resource`, `@mcp.prompt`를 사용해 직접 구현한다.


## 실습 실행 순서

아래 순서대로 실행하세요.

1. **FastMCP 설치**
2. **기본 import**
3. **샘플 SQLite DB 생성**
4. **MCP Server 정의**
5. **정적 Resource 정의**
6. **동적 Resource Template 정의**
7. **Prompt 정의**
8. **Client로 Resource 목록 조회**
9. **Client로 Resource 읽기**
10. **Client로 Prompt 목록 조회 및 렌더링**

권장:
- 메뉴에서 **런타임 → 모두 실행**
- 또는 위에서 아래로 한 셀씩 실행


## 1) FastMCP 설치

In [1]:
!pip -q install -U fastmcp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 728.6/728.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.3/204.3 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.4/96.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 152.3/152.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 19.8 MB/s eta 0:00:00


## 2) 기본 import

In [2]:
from fastmcp import FastMCP, Client
from fastmcp.prompts import Message
import sqlite3
import json
from pathlib import Path

print('import 완료')

import 완료


## 3) 샘플 SQLite DB 준비

이번 실습에서는 **DB 레코드도 Resource로 읽기 전용 제공**할 수 있음을 보여주기 위해,
간단한 메모 DB를 하나 만듭니다.

In [3]:
DB_PATH = Path('module4_resources_prompts.db')

def init_db():
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute(
        '''
        CREATE TABLE IF NOT EXISTS memos (
            id INTEGER PRIMARY KEY,
            title TEXT NOT NULL,
            content TEXT NOT NULL,
            author TEXT NOT NULL
        )
        '''
    )
    cur.execute('DELETE FROM memos')
    sample_rows = [
        (1, 'MCP 개요', 'MCP는 Host, Client, Server 구조를 사용합니다.', 'Instructor'),
        (2, 'Resources 개념', 'Resources는 문서와 DB 레코드 같은 읽기 전용 데이터를 제공합니다.', 'Instructor'),
        (3, 'Prompts 개념', 'Prompts는 재사용 가능한 지침 템플릿입니다.', 'Instructor'),
    ]
    cur.executemany('INSERT INTO memos (id, title, content, author) VALUES (?, ?, ?, ?)', sample_rows)
    conn.commit()
    conn.close()

init_db()
print(f'DB initialized: {DB_PATH.resolve()}')

DB initialized: /content/module4_resources_prompts.db


## 4) MCP Server 정의

FastMCP 서버 인스턴스를 만듭니다.
이번 실습에서는 Tool이 아니라 **Resources / Prompts** 중심으로 구성합니다.

In [4]:
mcp = FastMCP('Module4-Resources-Prompts-Demo')
print('MCP server created')

MCP server created


## 5) 정적 Resource 정의

정적 Resource는 고정된 URI로 접근하는 읽기 전용 데이터입니다.
예: 공지사항, 설정 파일, 강의 요약

In [5]:
@mcp.resource(
    'resource://course/notice',
    name='course_notice',
    description='강의 실습 공지사항',
    mime_type='text/plain',
    tags={'course', 'notice'}
)
def course_notice() -> str:
    """Module 4 실습 안내 문구를 제공합니다."""
    return 'Module 4 실습에서는 Resources와 Prompts의 차이를 이해하는 것이 핵심입니다.'

@mcp.resource(
    'resource://course/config',
    name='course_config',
    description='실습 환경 설정 정보',
    tags={'course', 'config'}
)
def course_config() -> dict:
    """실습 설정 정보를 JSON 형태로 제공합니다."""
    return {
        'module': 'Module 4',
        'topic': 'Resources and Prompts',
        'db_path': str(DB_PATH),
        'mode': 'colab-demo'
    }

print('정적 Resources 정의 완료')

정적 Resources 정의 완료


## 6) 동적 Resource Template 정의

Resource Template은 URI 안의 파라미터에 따라 다른 데이터를 만들어 반환합니다.
예: `memo://2` 또는 `memo://3` 같이 요청마다 다른 DB 레코드를 읽음

In [6]:
@mcp.resource(
    'memo://{memo_id}',
    name='memo_record',
    description='memo_id에 해당하는 메모 레코드를 읽기 전용으로 제공합니다.',
    tags={'db', 'memo'}
)
def memo_record(memo_id: str) -> dict:
    """DB에서 메모 1건을 읽어옵니다."""
    conn = sqlite3.connect(DB_PATH)
    cur = conn.cursor()
    cur.execute('SELECT id, title, content, author FROM memos WHERE id = ?', (int(memo_id),))
    row = cur.fetchone()
    conn.close()

    if row is None:
        return {
            'found': False,
            'memo_id': memo_id,
            'message': '해당 메모를 찾을 수 없습니다.'
        }

    return {
        'found': True,
        'id': row[0],
        'title': row[1],
        'content': row[2],
        'author': row[3]
    }

@mcp.resource(
    'symbol://{ticker}/summary',
    name='symbol_summary',
    description='종목 코드에 따른 요약 문장을 제공합니다.',
    tags={'dynamic', 'finance'}
)
def symbol_summary(ticker: str) -> dict:
    """간단한 종목 요약 예시를 반환합니다."""
    sample = {
        '005930': '삼성전자는 반도체와 전자제품 중심의 대표 기업입니다.',
        '000660': 'SK하이닉스는 메모리 반도체 중심 기업입니다.',
        '035420': 'NAVER는 플랫폼 및 인터넷 서비스 중심 기업입니다.'
    }
    return {
        'ticker': ticker,
        'summary': sample.get(ticker, '등록되지 않은 종목입니다.')
    }

print('동적 Resource Templates 정의 완료')

동적 Resource Templates 정의 완료


## 7) Prompt 정의

Prompt는 **LLM에게 전달할 재사용 가능한 지침 템플릿**입니다.
이번 실습에서는:
- 단일 메시지 Prompt
- 여러 메시지 Prompt

두 가지를 모두 만들어 봅니다.

In [7]:
@mcp.prompt
def summarize_resource(data_uri: str, audience: str = 'beginner') -> str:
    """특정 resource URI를 요약하도록 LLM에 지시하는 프롬프트를 생성합니다."""
    return (
        f'Please read the MCP resource at {data_uri} and summarize it for a {audience} audience. '
        'Return the answer in clear Korean bullet points.'
    )

@mcp.prompt
def teaching_assistant_prompt(topic: str, level: str = 'intro') -> list[Message]:
    """강의 도우미용 다중 메시지 프롬프트를 생성합니다."""
    return [
        Message(
            f'You are an instructor. Explain the topic "{topic}" for level={level}. '
            'Use examples and keep the explanation structured.'
        ),
        Message(
            '알겠습니다. 학습 목표, 핵심 개념, 예시, 실습 포인트 순서로 설명하겠습니다.',
            role='assistant'
        )
    ]

print('Prompts 정의 완료')

Prompts 정의 완료


## 8) Client Helper

버전 차이에 덜 민감하게 출력하기 위한 헬퍼 함수입니다.

In [8]:
def model_to_dict(obj):
    if hasattr(obj, 'model_dump'):
        return obj.model_dump()
    if isinstance(obj, dict):
        return obj
    result = {}
    for key in ['uri', 'uriTemplate', 'name', 'title', 'description', 'mimeType', 'arguments', '_meta']:
        if hasattr(obj, key):
            result[key] = getattr(obj, key)
    return result

def extract_text_content(items):
    extracted = []
    for item in items:
        if hasattr(item, 'text'):
            extracted.append({'mimeType': getattr(item, 'mimeType', None), 'text': item.text})
        elif hasattr(item, 'blob'):
            extracted.append({'mimeType': getattr(item, 'mimeType', None), 'blob_size': len(item.blob)})
        else:
            extracted.append(str(item))
    return extracted

def message_to_dict(msg):
    role = getattr(msg, 'role', None)
    content = getattr(msg, 'content', None)
    if hasattr(content, 'text'):
        content_value = content.text
    else:
        content_value = str(content)
    return {'role': role, 'content': content_value}

print('Helper 준비 완료')

Helper 준비 완료


## 9) Resource 목록 조회

정적 Resource와 동적 Resource Template을 각각 조회합니다.

In [9]:
async def show_resources():
    client = Client(mcp)
    async with client:
        resources = await client.list_resources()
        templates = await client.list_resource_templates()

        print('=== Static Resources ===')
        for r in resources:
            print(json.dumps(model_to_dict(r), ensure_ascii=False, indent=2, default=str))

        print('\n=== Resource Templates ===')
        for t in templates:
            print(json.dumps(model_to_dict(t), ensure_ascii=False, indent=2, default=str))

await show_resources()

=== Static Resources ===
{
  "name": "course_notice",
  "title": null,
  "uri": "resource://course/notice",
  "description": "강의 실습 공지사항",
  "mimeType": "text/plain",
  "size": null,
  "icons": null,
  "annotations": null,
  "meta": {
    "fastmcp": {
      "tags": [
        "course",
        "notice"
      ]
    }
  }
}
{
  "name": "course_config",
  "title": null,
  "uri": "resource://course/config",
  "description": "실습 환경 설정 정보",
  "mimeType": "text/plain",
  "size": null,
  "icons": null,
  "annotations": null,
  "meta": {
    "fastmcp": {
      "tags": [
        "config",
        "course"
      ]
    }
  }
}

=== Resource Templates ===
{
  "name": "memo_record",
  "title": null,
  "uriTemplate": "memo://{memo_id}",
  "description": "memo_id에 해당하는 메모 레코드를 읽기 전용으로 제공합니다.",
  "mimeType": "text/plain",
  "icons": null,
  "annotations": null,
  "meta": {
    "fastmcp": {
      "tags": [
        "db",
        "memo"
      ]
    }
  }
}
{
  "name": "symbol_summary",
  "title": null,
  "

## 10) Resource 읽기

이제 실제로 Resource를 읽어 봅니다.

- 정적 Resource: `resource://course/notice`
- 동적 Template: `memo://2`
- 동적 Template: `symbol://005930/summary`


In [10]:
async def read_resources_demo():
    client = Client(mcp)
    async with client:
        notice = await client.read_resource('resource://course/notice')
        memo2 = await client.read_resource('memo://2')
        symbol = await client.read_resource('symbol://005930/summary')

        print('=== resource://course/notice ===')
        print(json.dumps(extract_text_content(notice), ensure_ascii=False, indent=2))

        print('\n=== memo://2 ===')
        print(json.dumps(extract_text_content(memo2), ensure_ascii=False, indent=2))

        print('\n=== symbol://005930/summary ===')
        print(json.dumps(extract_text_content(symbol), ensure_ascii=False, indent=2))

await read_resources_demo()

=== resource://course/notice ===
[
  {
    "mimeType": "text/plain",
    "text": "Module 4 실습에서는 Resources와 Prompts의 차이를 이해하는 것이 핵심입니다."
  }
]

=== memo://2 ===
[
  {
    "mimeType": "application/json",
    "text": "{\"found\": true, \"id\": 2, \"title\": \"Resources \\uac1c\\ub150\", \"content\": \"Resources\\ub294 \\ubb38\\uc11c\\uc640 DB \\ub808\\ucf54\\ub4dc \\uac19\\uc740 \\uc77d\\uae30 \\uc804\\uc6a9 \\ub370\\uc774\\ud130\\ub97c \\uc81c\\uacf5\\ud569\\ub2c8\\ub2e4.\", \"author\": \"Instructor\"}"
  }
]

=== symbol://005930/summary ===
[
  {
    "mimeType": "application/json",
    "text": "{\"ticker\": \"005930\", \"summary\": \"\\uc0bc\\uc131\\uc804\\uc790\\ub294 \\ubc18\\ub3c4\\uccb4\\uc640 \\uc804\\uc790\\uc81c\\ud488 \\uc911\\uc2ec\\uc758 \\ub300\\ud45c \\uae30\\uc5c5\\uc785\\ub2c8\\ub2e4.\"}"
  }
]


## 11) Prompt 목록 조회

Prompt는 서버가 제공하는 **재사용 가능한 메시지 템플릿**입니다.
먼저 어떤 Prompt가 있는지 확인합니다.

In [11]:
async def show_prompts():
    client = Client(mcp)
    async with client:
        prompts = await client.list_prompts()
        print('=== Prompts ===')
        for p in prompts:
            print(json.dumps(model_to_dict(p), ensure_ascii=False, indent=2, default=str))

await show_prompts()

=== Prompts ===
{
  "name": "summarize_resource",
  "title": null,
  "description": "특정 resource URI를 요약하도록 LLM에 지시하는 프롬프트를 생성합니다.",
  "arguments": [
    {
      "name": "data_uri",
      "description": null,
      "required": true
    },
    {
      "name": "audience",
      "description": null,
      "required": false
    }
  ],
  "icons": null,
  "meta": {
    "fastmcp": {
      "tags": []
    }
  }
}
{
  "name": "teaching_assistant_prompt",
  "title": null,
  "description": "강의 도우미용 다중 메시지 프롬프트를 생성합니다.",
  "arguments": [
    {
      "name": "topic",
      "description": null,
      "required": true
    },
    {
      "name": "level",
      "description": null,
      "required": false
    }
  ],
  "icons": null,
  "meta": {
    "fastmcp": {
      "tags": []
    }
  }
}


## 12) Prompt 렌더링

이제 Prompt를 실제로 렌더링해서 어떤 메시지가 만들어지는지 확인합니다.

In [12]:
async def render_prompts_demo():
    client = Client(mcp)
    async with client:
        p1 = await client.get_prompt(
            'summarize_resource',
            {'data_uri': 'memo://2', 'audience': 'manager'}
        )
        p2 = await client.get_prompt(
            'teaching_assistant_prompt',
            {'topic': 'MCP Resources와 Prompts', 'level': 'intro'}
        )

        print('=== Prompt: summarize_resource ===')
        for msg in p1.messages:
            print(json.dumps(message_to_dict(msg), ensure_ascii=False, indent=2))

        print('\n=== Prompt: teaching_assistant_prompt ===')
        for msg in p2.messages:
            print(json.dumps(message_to_dict(msg), ensure_ascii=False, indent=2))

await render_prompts_demo()

=== Prompt: summarize_resource ===
{
  "role": "user",
  "content": "Please read the MCP resource at memo://2 and summarize it for a manager audience. Return the answer in clear Korean bullet points."
}

=== Prompt: teaching_assistant_prompt ===
{
  "role": "user",
  "content": "You are an instructor. Explain the topic \"MCP Resources와 Prompts\" for level=intro. Use examples and keep the explanation structured."
}
{
  "role": "assistant",
  "content": "알겠습니다. 학습 목표, 핵심 개념, 예시, 실습 포인트 순서로 설명하겠습니다."
}


## 13) 실습 정리

이번 실습에서 확인한 핵심은 다음과 같습니다.

1. **Resources**는 읽기 전용 데이터 제공 방식이다.
2. Resource는 정적 URI와 동적 URI 템플릿 둘 다 만들 수 있다.
3. DB 레코드 같은 데이터도 Resource로 제공할 수 있다.
4. **Prompts**는 재사용 가능한 메시지 템플릿이다.
5. Prompt는 단일 메시지 또는 다중 메시지 대화 흐름으로 만들 수 있다.
6. Tool은 행동(action)을 수행하고, Resource는 데이터를 읽게 하며, Prompt는 LLM 지침을 재사용하게 한다.
